# Prompt 14: HCP Terraform — Workspaces, Runs, and State

**Terraform Associate (004) — Objective 8: HCP Terraform (Second Largest Exam Topic)**

This notebook covers HCP Terraform (formerly Terraform Cloud) core functionality: workspaces, run lifecycle, CLI-driven and VCS-driven workflows, the `cloud` block, variable sets, remote state data sources, run triggers, and projects.

---

## Topics Covered

1. What HCP Terraform Is
2. HCP Terraform vs. Terraform Community Edition
3. HCP Terraform Workspaces
4. Run Lifecycle
5. CLI-Driven Workflow and the `cloud` Block
6. `cloud` Block vs. `backend "remote"`
7. VCS-Driven Workflow
8. `terraform login` and Authentication
9. Migrating State to HCP Terraform
10. Variable Sets
11. Remote State Data Source (`terraform_remote_state`)
12. Run Triggers and Projects
13. Exam-Style Questions
14. Real-World Scenarios

---

## 1. What HCP Terraform Is

**HCP Terraform** (HashiCorp Cloud Platform Terraform, formerly **Terraform Cloud**) is HashiCorp's hosted SaaS service that adds collaboration, governance, and operational features on top of the open-source Terraform CLI.

### Core Capabilities

| Capability | Description |
|---|---|
| **Remote state storage** | State files stored securely in HCP, encrypted at rest, versioned |
| **Remote execution** | Plan and apply run on HCP-managed workers — consistent, isolated environment |
| **Team collaboration** | Multiple users with role-based permissions per workspace |
| **Policy enforcement** | Sentinel and OPA policies gate applies |
| **Audit logging** | Full history of who ran what and when |
| **Drift detection** | Health assessments run on a schedule |
| **Variable management** | Centralized variables and variable sets across workspaces |
| **SSO / SAML** | Enterprise-grade authentication |

> **Exam tip**: HCP Terraform is the **second largest topic** on the Terraform Associate exam. Expect several questions on workspaces, run lifecycle, variable sets, `cloud` block, and `terraform_remote_state`.

---

## 2. HCP Terraform vs. Terraform Community Edition

| Feature | Community Edition (CLI) | HCP Terraform |
|---|---|---|
| State storage | Local `terraform.tfstate` | Remote, encrypted, versioned |
| State locking | Backend-dependent (S3+DynamoDB, etc.) | Built-in, automatic |
| Plan/apply execution | Local machine | Managed workers (remote) |
| Team collaboration | None built-in | Role-based access, team tokens |
| Policy enforcement | None | Sentinel, OPA |
| Audit logging | None | Full organization-level audit log |
| Drift detection | Manual (`plan -refresh-only`) | Automatic health assessments |
| VCS integration | None | GitHub, GitLab, Bitbucket, Azure DevOps |
| SSO / SAML | None | Supported (Plus/Enterprise) |
| Cost | Free (open source) | Free tier + paid plans |

> **Key exam point**: The Community Edition is a **local CLI tool only**. HCP Terraform adds the hosted platform layer that makes Terraform suitable for teams and organizations.

---

## 3. HCP Terraform Workspaces

### Definition

In HCP Terraform, a **workspace** is an isolated unit that contains:

- Its own **state file** (versioned)
- Its own **variable store** (Terraform and environment variables)
- Its own **run history** (all plans and applies)
- Its own **settings** (Terraform version, execution mode, auto-apply)

### ⚠️ HCP Workspaces ≠ Local Workspaces

| | HCP Terraform Workspaces | Local `terraform workspace` |
|---|---|---|
| **What they are** | Full isolated environments in HCP | Named state files in the same config directory |
| **State** | Separate, stored in HCP | Separate, stored in `terraform.tfstate.d/` locally |
| **Variables** | Each workspace has its own variable store | All workspaces share the same config |
| **Collaboration** | Yes — multi-user, team-based | No |
| **VCS integration** | Yes | No |

> **Critical exam point**: HCP Terraform workspaces and local `terraform workspace` commands are **completely different concepts** — they are NOT interchangeable.

### Execution Modes

| Mode | Who Runs Plan/Apply | State Location | Use Case |
|---|---|---|---|
| **Remote** | HCP managed workers | HCP | Standard — consistent environment |
| **Local** | CLI on your machine | HCP | Needed when CLI can reach resources but HCP workers cannot |
| **Agent** | Customer-managed agent | HCP | Private networks, on-prem, VPC-isolated resources |

### Key Workspace Settings

| Setting | Description |
|---|---|
| Terraform version | Lock workspace to specific Terraform version |
| Execution mode | Remote / Local / Agent |
| Auto-apply | Automatically apply after a successful plan (no confirmation needed) |
| Run triggers | Automatically queue this workspace's run when another workspace applies |
| Remote state sharing | Allow other workspaces to read this workspace's outputs |

---

## 4. Run Lifecycle

A **run** in HCP Terraform is a single plan+apply cycle. Every run follows this lifecycle:

```
Trigger (VCS push, CLI, API, run trigger)
        ↓
    Plan
  (Terraform generates execution plan)
        ↓
  Policy Check
  (Sentinel / OPA policies evaluate the plan)
        ↓
  Apply Confirmation
  (Team member reviews and approves — unless auto-apply is on)
        ↓
    Apply
  (Infrastructure changes are made)
        ↓
  State Updated
```

### Run Types

| Run Type | Description |
|---|---|
| **Plan + Apply** | Full run — creates/modifies/destroys resources |
| **Plan only (speculative)** | VCS PR triggers — shows what would change, no apply |
| **Destroy** | Runs `terraform destroy` to remove all resources |
| **Refresh only** | Updates state without changing resources |

### Run Statuses

| Status | Meaning |
|---|---|
| Pending | Queued, waiting for worker |
| Planning | `terraform plan` in progress |
| Cost estimation | Estimating cost impact (if enabled) |
| Policy check | Sentinel/OPA evaluating the plan |
| Needs confirmation | Plan complete, waiting for human approval |
| Applying | `terraform apply` in progress |
| Applied | Completed successfully |
| Errored | Failed at some stage |

> **Exam tip**: Policy check happens BETWEEN plan and apply. If a hard-mandatory policy fails, the run is blocked — no human can override it.

---

## 5. CLI-Driven Workflow and the `cloud` Block

The CLI-driven workflow lets you use familiar `terraform plan` and `terraform apply` commands from your local terminal while HCP Terraform manages state and (optionally) executes the run remotely.

### Setup Steps

```
1. Create organization + workspace in HCP Terraform UI
        ↓
2. Add cloud block to terraform.tf
        ↓
3. terraform login
        ↓
4. terraform init  (connects to HCP workspace)
        ↓
5. terraform plan / terraform apply  (runs on HCP workers)
```

### `cloud` Block Syntax

```hcl
terraform {
  cloud {
    organization = "my-org"

    workspaces {
      name = "my-app-production"  # single workspace by name
    }
  }
}
```

Or use **tag-based workspace selection** (unique to `cloud` block):

```hcl
terraform {
  cloud {
    organization = "my-org"

    workspaces {
      tags = ["app:web", "env:production"]  # all matching workspaces
    }
  }
}
```

### `cloud` Block Arguments

| Argument | Required | Description |
|---|---|---|
| `organization` | Yes | HCP Terraform organization name |
| `workspaces.name` | One of | Target a single workspace by name |
| `workspaces.tags` | One of | Target all workspaces with matching tags |
| `hostname` | No | For Terraform Enterprise (default: `app.terraform.io`) |
| `token` | No | Auth token (prefer `terraform login` or `TF_TOKEN_*` env var) |

In [ ]:
# cloud block configuration examples

cloud_block_single_workspace = '''
# terraform.tf — single named workspace
terraform {
  required_version = ">= 1.1"

  required_providers {
    aws = {
      source  = "hashicorp/aws"
      version = "~> 5.0"
    }
  }

  cloud {
    organization = "acme-corp"

    workspaces {
      name = "web-app-production"
    }
  }
}
'''

cloud_block_tags = '''
# terraform.tf — tag-based workspace selection
terraform {
  cloud {
    organization = "acme-corp"

    workspaces {
      tags = ["project:web-app"]  # all workspaces tagged project:web-app
    }
  }
}

# When using tags, select workspace before running:
# $ terraform workspace list
# $ terraform workspace select web-app-production
# $ terraform plan
'''

print(cloud_block_single_workspace)
print(cloud_block_tags)

In [ ]:
# CLI-driven workflow commands

cli_workflow = '''
# Step 1: Authenticate to HCP Terraform
$ terraform login
# Opens browser → log in → token stored at:
# ~/.terraform.d/credentials.tfrc.json (Linux/Mac)
# %APPDATA%/terraform.d/credentials.tfrc.json (Windows)

# Step 2: Initialize — connects to HCP workspace
$ terraform init
# Initializing Terraform Cloud...
# Terraform Cloud has been successfully initialized!

# Step 3: Plan — runs on HCP workers (remote execution mode)
$ terraform plan
# Running plan in Terraform Cloud. Output will stream here.
# Waiting for the plan to start...
# ...
# Plan: 3 to add, 0 to change, 0 to destroy.

# Step 4: Apply — streams output from HCP worker
$ terraform apply
# Running apply in Terraform Cloud. Output will stream here.
# Do you want to perform these actions in workspace "web-app-production"?
#   Terraform will perform the actions described above.
#   Only 'yes' will be accepted to approve.
# Enter a value: yes
'''

print(cli_workflow)

---

## 6. `cloud` Block vs. `backend "remote"`

| Feature | `cloud` block | `backend "remote"` |
|---|---|---|
| **Available since** | Terraform 1.1 | Older Terraform versions |
| **Status** | **Preferred** for HCP Terraform | Deprecated for HCP Terraform usage |
| **Tag-based workspace filtering** | Yes | No |
| **Feature completeness** | Full | Limited |
| **Still functional** | Yes | Yes (but use `cloud` block for new configs) |
| **Terraform Enterprise** | Supported | Supported |

### Legacy `backend "remote"` (for reference)

```hcl
# OLD approach — still works but prefer cloud block
terraform {
  backend "remote" {
    organization = "acme-corp"

    workspaces {
      name = "web-app-production"
    }
  }
}
```

> **Exam tip**: If you see `backend "remote"` on the exam, it is the legacy way to connect to HCP Terraform. The `cloud` block is the **current preferred approach** and is described as more feature-complete in official HashiCorp documentation. Both connect to HCP Terraform but `cloud` block is the answer to prefer.

---

## 7. VCS-Driven Workflow

The VCS-driven workflow connects an HCP Terraform workspace directly to a version control repository. This is the most common production workflow.

### Supported VCS Providers

- GitHub / GitHub Enterprise
- GitLab / GitLab Self-Managed
- Bitbucket Cloud / Bitbucket Server
- Azure DevOps

### How It Works

```
Developer opens Pull Request
        ↓
HCP Terraform runs SPECULATIVE PLAN
(shows what would change — no apply)
Plan result posted as PR check/comment
        ↓
PR is reviewed and MERGED to default branch
        ↓
HCP Terraform queues a PLAN + APPLY run
        ↓
Run applies (with confirmation or auto-apply)
```

### Key Terms

| Term | Meaning |
|---|---|
| **Speculative plan** | A read-only plan triggered by a PR/MR — never applies, used for review |
| **Working directory** | Which directory in the repo the workspace uses as its root module |
| **Trigger patterns** | File paths that, when changed, trigger a run (avoids runs for unrelated changes) |
| **Auto-apply** | Workspace setting to apply immediately after a successful plan without human confirmation |

> **Exam tip**: In VCS-driven workflow, you do NOT run `terraform apply` locally — all applies happen in HCP Terraform triggered by VCS events. The CLI-driven workflow is when you run commands locally.

---

## 8. `terraform login` and Authentication

### `terraform login`

```bash
terraform login              # authenticate to app.terraform.io (HCP Terraform)
terraform login app.terraform.io  # explicit (same as above)
terraform login my-tfe.example.com  # Terraform Enterprise instance
```

**What happens**:
1. Opens a browser window to HCP Terraform
2. User logs in and generates an API token
3. Token is stored at `~/.terraform.d/credentials.tfrc.json`

### `terraform logout`

```bash
terraform logout  # removes stored token
```

### Alternative Authentication Methods

| Method | How |
|---|---|
| `terraform login` | Interactive browser flow |
| `TF_TOKEN_app_terraform_io` env var | Set token as environment variable (CI/CD) |
| `credentials.tfrc.json` | Manually write token to credentials file |

> **Key exam point**: Token is stored at `~/.terraform.d/credentials.tfrc.json`. In CI/CD pipelines, use the `TF_TOKEN_app_terraform_io` environment variable instead of running `terraform login` interactively.

In [ ]:
# terraform login and credentials file example

login_example = '''
# Authenticate to HCP Terraform
$ terraform login

Terraform will request an API token for app.terraform.io using your browser.

If login is successful, Terraform will store the token in plain text in
the following file for use by subsequent commands:
    /home/user/.terraform.d/credentials.tfrc.json

Do you want to proceed? (yes/no) yes
# Opens browser...
# After approval:
Retrieved token for user pbaletkeman

---------------------------------------------------------------------------------
You are now logged in. The token stored in credentials.tfrc.json will be used
for all future Terraform commands for app.terraform.io.
'''

credentials_file = '''
# ~/.terraform.d/credentials.tfrc.json
{
  "credentials": {
    "app.terraform.io": {
      "token": "<your-api-token-here>"
    }
  }
}
'''

ci_auth = '''
# CI/CD — set token as environment variable (no interactive login needed)
export TF_TOKEN_app_terraform_io="<your-api-token>"
terraform init
terraform plan
'''

print(login_example)
print(credentials_file)
print(ci_auth)

---

## 9. Migrating State to HCP Terraform

If you have an existing project with local state or an S3 backend and want to migrate to HCP Terraform:

```
1. Create workspace in HCP Terraform UI
        ↓
2. Add cloud block to terraform.tf
        ↓
3. terraform login  (if not already authenticated)
        ↓
4. terraform init
   Terraform detects existing state and prompts:
   "Do you wish to proceed? (yes/no)"
        ↓
5. State is copied to HCP Terraform workspace
        ↓
6. Verify with terraform show or HCP Terraform UI
```

### Important Notes

- `terraform init` handles the migration automatically — it detects existing state and prompts for confirmation
- The local `terraform.tfstate` file is **not deleted** automatically — you should delete it manually after verifying the migration
- If migrating from another backend (e.g., S3), the backend config must be replaced with the `cloud` block before running `init`

In [ ]:
# Migration example: local state → HCP Terraform

migration_config = '''
# Before migration: no backend block (local state)
# terraform.tf
terraform {
  required_version = ">= 1.1"
}

# After: add cloud block
# terraform.tf
terraform {
  required_version = ">= 1.1"

  cloud {
    organization = "acme-corp"

    workspaces {
      name = "my-project"
    }
  }
}
'''

migration_init = '''
$ terraform init

Initializing Terraform Cloud...
Do you wish to proceed?
  As part of migrating to Terraform Cloud, Terraform can optionally copy your
  current workspace state to the configured Terraform Cloud workspace.

  Answer "yes" to copy the latest state snapshot to the configured
  Terraform Cloud workspace.
  Answer "no" to ignore the existing state and just activate the configured
  Terraform Cloud workspace with its existing state, if any.

  Should Terraform migrate your existing state? Enter a value: yes

Successfully migrated your state to Terraform Cloud!
'''

print(migration_config)
print(migration_init)

---

## 10. Variable Sets

**Variable sets** are reusable collections of Terraform or environment variables that can be applied to multiple workspaces — or even the entire organization.

### Why Variable Sets?

Without variable sets, every workspace would need the same AWS credentials set individually. With variable sets:

```
Variable Set: "AWS Production Credentials"
  AWS_ACCESS_KEY_ID  = ****
  AWS_SECRET_ACCESS_KEY = ****
  AWS_REGION = us-east-1

Applied to:
  ✓ web-app-production
  ✓ api-service-production
  ✓ data-pipeline-production
```

### Variable Types in HCP Terraform

| Type | Prefix | Use Case |
|---|---|---|
| **Terraform variable** | `var.*` | Inputs to Terraform configuration |
| **Environment variable** | `env.*` | Shell environment, provider credentials |

### Variable Precedence

When multiple sources define the same variable, workspace-level variables **override** variable set values.

```
Priority (highest to lowest):
1. Workspace-specific variable
2. Variable set assigned to workspace
3. Global variable set (applied to all workspaces)
```

> **Exam tip**: Variable sets are created at the **organization level** and then applied to specific workspaces or globally. They are ideal for shared credentials, common tags, and organization-wide defaults.

In [ ]:
# Variable set concept — no HCL needed, configured in HCP UI
# But you can reference variables in code normally

variable_set_usage = '''
# Variables defined in an HCP variable set are injected into
# the workspace run environment automatically.

# In your Terraform code, reference them normally:
provider "aws" {
  # AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY come from the
  # "AWS Production Credentials" variable set as environment variables
  # No explicit credential configuration needed in code!
  region = var.aws_region  # terraform variable from variable set
}

variable "aws_region" {
  type    = string
  default = "us-east-1"  # overridden by variable set value
}

# Variable set defined in HCP Terraform UI:
#
# Variable Set: "AWS Production Credentials"
#   Type: Environment Variable  Name: AWS_ACCESS_KEY_ID     Value: AKIA... (sensitive)
#   Type: Environment Variable  Name: AWS_SECRET_ACCESS_KEY Value: ****  (sensitive)
#   Type: Terraform Variable    Name: aws_region            Value: us-east-1
#
# Applied to workspaces: all production workspaces
'''

print(variable_set_usage)

---

## 11. Remote State Data Source (`terraform_remote_state`)

The `terraform_remote_state` data source lets one workspace read the **output values** of another workspace's state. This enables workspace-to-workspace data sharing without tight coupling.

### Use Case

```
Workspace A (networking)          Workspace B (application)
  outputs:
    vpc_id = "vpc-0abc123"    →   reads vpc_id from Workspace A
    subnet_ids = [...]        →   reads subnet_ids from Workspace A
```

### Syntax

```hcl
data "terraform_remote_state" "networking" {
  backend = "remote"

  config = {
    organization = "acme-corp"
    workspaces = {
      name = "networking-production"
    }
  }
}

# Reference outputs from the remote workspace
resource "aws_instance" "app" {
  subnet_id = data.terraform_remote_state.networking.outputs.public_subnet_id
  vpc_security_group_ids = [data.terraform_remote_state.networking.outputs.app_sg_id]
}
```

### Requirements

- The source workspace must have **remote state sharing enabled** (or be shared with the consuming workspace)
- Only **output values** are accessible — not individual resource attributes
- The source workspace's outputs must be declared with `output` blocks

> **Exam tip**: `terraform_remote_state` only exposes `outputs` — not the entire state. If a value isn't declared as an output in the source workspace, it cannot be read remotely.

In [ ]:
# Full terraform_remote_state example

networking_workspace_outputs = '''
# networking workspace — outputs.tf
output "vpc_id" {
  value = aws_vpc.main.id
}

output "public_subnet_ids" {
  value = aws_subnet.public[*].id
}

output "app_security_group_id" {
  value = aws_security_group.app.id
}
'''

application_workspace_remote_state = '''
# application workspace — main.tf
data "terraform_remote_state" "networking" {
  backend = "remote"

  config = {
    organization = "acme-corp"
    workspaces = {
      name = "networking-production"
    }
  }
}

resource "aws_instance" "app" {
  ami           = "ami-0c55b159cbfafe1f0"
  instance_type = "t3.medium"

  # Use outputs from the networking workspace
  subnet_id              = data.terraform_remote_state.networking.outputs.public_subnet_ids[0]
  vpc_security_group_ids = [data.terraform_remote_state.networking.outputs.app_security_group_id]

  tags = {
    Name   = "app-server"
    VPC_ID = data.terraform_remote_state.networking.outputs.vpc_id
  }
}
'''

print(networking_workspace_outputs)
print(application_workspace_remote_state)

---

## 12. Run Triggers and Projects

### Run Triggers

**Run triggers** automatically queue a run in a downstream workspace when an upstream workspace successfully applies.

```
Workspace: networking (source)
  Apply completes
        ↓  (run trigger)
Workspace: application (downstream)
  Run is automatically queued
```

**Use cases**:
- Networking workspace finishes → automatically trigger application workspace
- Shared module update → propagate to all dependent workspaces
- Multi-tier deployments: infrastructure → platform → application

**Configuration**: Set in HCP Terraform workspace settings UI or via API. One workspace can have multiple source workspaces as triggers.

### Projects

**Projects** are a way to organize multiple workspaces within an organization.

| Feature | Description |
|---|---|
| **Grouping** | Logical containers for related workspaces |
| **Access control** | Grant team permissions at the project level (applies to all workspaces in the project) |
| **`project_id`** | Reference a project when creating workspaces via API/provider |

```
Organization: acme-corp
  ├── Project: web-application
  │     ├── web-app-production
  │     ├── web-app-staging
  │     └── web-app-development
  ├── Project: data-platform
  │     ├── data-pipeline-production
  │     └── data-warehouse-production
  └── Project: networking
        ├── networking-production
        └── networking-staging
```

> **Exam tip**: Projects control access for all workspaces within them. This is more scalable than setting permissions on individual workspaces when you have many workspaces.

---

## 13. Exam-Style Questions

### Question 1

A team is using Terraform 1.3 and wants to connect to HCP Terraform. A developer proposes using `backend "remote"`. Another suggests using the `cloud` block. Which is correct?

**A.** Both work, but `backend "remote"` is the recommended approach for all Terraform versions  
**B.** The `cloud` block is only available in Terraform 1.5 and later  
**C.** Both work, but the `cloud` block is the preferred approach — it supports tag-based workspace filtering and is more feature-complete  
**D.** `backend "remote"` is deprecated and will no longer connect to HCP Terraform  

<details><summary>Answer</summary>

**C** is correct. Both `backend "remote"` and the `cloud` block can connect to HCP Terraform, but the `cloud` block (available since Terraform **1.1**) is the **preferred** approach. It supports tag-based workspace filtering, which `backend "remote"` does not, and is more feature-complete.

**A** is wrong — `backend "remote"` is the legacy approach.

**B** is wrong — the `cloud` block was introduced in Terraform 1.1, not 1.5.

**D** is wrong — `backend "remote"` still works; it is deprecated for HCP Terraform usage in favor of the `cloud` block, but it has not been removed.

</details>

---

### Question 2

An HCP Terraform workspace has the following configuration: execution mode is set to **remote**, and auto-apply is **disabled**. What happens when a developer pushes a commit to the connected VCS repository's main branch?

**A.** Terraform applies the changes immediately on the HCP worker without any confirmation  
**B.** Terraform queues a plan on HCP workers; after the plan completes, a team member must confirm before the apply runs  
**C.** The developer must run `terraform apply` from their local CLI to execute the changes  
**D.** A speculative plan runs; this plan can never be applied — a separate run must be triggered manually  

<details><summary>Answer</summary>

**B** is correct. In the VCS-driven workflow with remote execution mode and auto-apply disabled: pushing to main queues a full plan on HCP workers. After the plan completes, a team member must review the plan output in the HCP Terraform UI and click "Confirm & Apply" before the apply executes.

**A** is wrong — that would be the behavior if auto-apply were **enabled**.

**C** is wrong — in VCS-driven workflow with remote execution mode, `terraform apply` is not run locally; the apply happens in HCP.

**D** is wrong — a speculative plan is triggered by a PR/MR (not a merge to main). A merge to main triggers a full plan+apply run.

</details>

---

### Question 3

A `networking` workspace exports `vpc_id` as an output. An `application` workspace needs to use this value to place an EC2 instance in the correct VPC. Which is the correct way to access this value from the `application` workspace?

**A.** Use `terraform_remote_state` data source to read outputs from the `networking` workspace  
**B.** Copy the `vpc_id` value manually into the `application` workspace's variables  
**C.** Use a `depends_on` meta-argument pointing to the `networking` workspace  
**D.** Use `data.external` to query the HCP Terraform API for the networking workspace's state  

<details><summary>Answer</summary>

**A** is correct. The `terraform_remote_state` data source is specifically designed to read output values from another workspace's state. This is the official, supported mechanism for cross-workspace data sharing in HCP Terraform.

**B** is a workaround that works but is not best practice — it requires manual updates and is error-prone.

**C** is wrong — `depends_on` works within a single configuration, not across HCP Terraform workspaces.

**D** is overly complex and fragile — `terraform_remote_state` is the correct tool.

</details>

---

### Question 4

A platform team manages 20 HCP Terraform workspaces across development, staging, and production environments. All workspaces need the same AWS provider credentials. What is the most efficient way to manage this in HCP Terraform?

**A.** Set `AWS_ACCESS_KEY_ID` and `AWS_SECRET_ACCESS_KEY` as environment variables in each of the 20 workspaces individually  
**B.** Hard-code credentials in the provider block in each workspace's configuration  
**C.** Create a **variable set** containing the AWS credentials and apply it to all 20 workspaces  
**D.** Store credentials in a Git repository that all workspaces reference  

<details><summary>Answer</summary>

**C** is correct. **Variable sets** are the HCP Terraform feature designed exactly for this use case. Create one variable set with the AWS credentials, mark them as sensitive, and apply the set to all workspaces (or all workspaces in a project). When credentials rotate, update one variable set — all workspaces automatically use the new values.

**A** is technically correct but highly inefficient and error-prone at scale — 20 manual updates on every credential rotation.

**B** is a critical security risk — credentials should never be hard-coded in configuration files.

**D** is a critical security risk — credentials must never be stored in version control.

</details>

---

## 14. Real-World Scenarios

<details><summary>Scenario 1: Migrating a Team from Local State to HCP Terraform</summary>

**Situation**: A startup has been using Terraform with local state files. Two developers have accidentally run conflicting applies because there is no locking. The CTO has approved an HCP Terraform free tier account. The team needs to migrate their existing infrastructure state to HCP Terraform without disrupting running resources.

**HCL Configuration Changes**:

```hcl
# BEFORE: terraform.tf (no backend — local state)
terraform {
  required_version = ">= 1.3"
  required_providers {
    aws = {
      source  = "hashicorp/aws"
      version = "~> 5.0"
    }
  }
}

# AFTER: terraform.tf (cloud block added)
terraform {
  required_version = ">= 1.3"
  required_providers {
    aws = {
      source  = "hashicorp/aws"
      version = "~> 5.0"
    }
  }

  cloud {
    organization = "startup-co"
    workspaces {
      name = "production"
    }
  }
}
```

**CLI Steps**:

```bash
# 1. Create organization and workspace in HCP Terraform UI first

# 2. Authenticate
terraform login

# 3. Init — Terraform prompts to migrate existing state
terraform init
# "Should Terraform migrate your existing state? Enter a value: yes"

# 4. Verify migration — state now in HCP
terraform state list

# 5. Clean up local state file (after confirming)
rm terraform.tfstate terraform.tfstate.backup
```

**Expected Outcome**: State is now stored in HCP Terraform with automatic locking, versioning, and encryption. No resources were modified or recreated.

**Exam Sub-Objective**: Migrate state to HCP Terraform; understand `cloud` block and `terraform init` migration flow (Objective 8).

</details>

---

<details><summary>Scenario 2: Sharing AWS Credentials Across Workspaces via Variable Sets</summary>

**Situation**: An organization has 15 Terraform workspaces in HCP Terraform, all deploying to the same AWS account. When the AWS IAM access keys were rotated, a developer had to update credentials in each workspace individually — taking 45 minutes and missing 3 workspaces, causing failed runs. The team wants to centralize credential management.

**Solution — Variable Set Setup**:

In the HCP Terraform UI (Organization → Variable Sets → New Variable Set):

```
Variable Set Name: "AWS Production Credentials"
Scope: Apply to specific projects (select all relevant projects)

Variables:
  ┌─────────────────────────┬──────────────┬────────────────┬───────────┐
  │ Key                     │ Value        │ Category       │ Sensitive │
  ├─────────────────────────┼──────────────┼────────────────┼───────────┤
  │ AWS_ACCESS_KEY_ID       │ AKIA...      │ env            │ Yes       │
  │ AWS_SECRET_ACCESS_KEY   │ ****         │ env            │ Yes       │
  │ AWS_DEFAULT_REGION      │ us-east-1    │ env            │ No        │
  └─────────────────────────┴──────────────┴────────────────┴───────────┘
```

**Terraform Code** (no credential config needed in provider):

```hcl
provider "aws" {
  # Credentials injected automatically from variable set
  # No access_key or secret_key needed here
}
```

**Expected Outcome**: When keys are rotated next time, update one variable set — all 15 workspaces automatically use the new credentials on their next run. The 45-minute manual process becomes a 2-minute update.

**Exam Sub-Objective**: Use variable sets to share common variables across workspaces (Objective 8).

</details>

---

<details><summary>Scenario 3: Cross-Workspace Data Sharing with terraform_remote_state</summary>

**Situation**: A large company manages their infrastructure with two separate workspaces: `networking` (VPC, subnets, security groups) and `application` (EC2, RDS, load balancer). The application workspace needs the VPC and subnet IDs from the networking workspace. Hardcoding IDs would make the config brittle.

**Networking workspace — outputs.tf**:

```hcl
output "vpc_id" {
  value       = aws_vpc.main.id
  description = "The ID of the main VPC"
}

output "private_subnet_ids" {
  value       = aws_subnet.private[*].id
  description = "IDs of private subnets"
}

output "app_security_group_id" {
  value       = aws_security_group.app.id
}
```

**Application workspace — main.tf**:

```hcl
data "terraform_remote_state" "networking" {
  backend = "remote"

  config = {
    organization = "acme-corp"
    workspaces = {
      name = "networking-production"
    }
  }
}

resource "aws_db_instance" "app" {
  engine         = "mysql"
  instance_class = "db.t3.medium"

  # Dynamically reads from networking workspace outputs
  db_subnet_group_name   = aws_db_subnet_group.app.name
  vpc_security_group_ids = [data.terraform_remote_state.networking.outputs.app_security_group_id]
}

resource "aws_db_subnet_group" "app" {
  subnet_ids = data.terraform_remote_state.networking.outputs.private_subnet_ids
}
```

**HCP Terraform Setup Required**:
- Enable "Remote state sharing" in the `networking-production` workspace settings
- Share with the `application-production` workspace (or globally within the organization)

**Expected Outcome**: The application workspace always uses the current subnet and security group IDs from the networking workspace. When networking changes subnets, the application workspace automatically picks up the new values on its next plan.

**Exam Sub-Objective**: Use `terraform_remote_state` to share outputs across workspaces (Objective 8).

</details>

---

<details><summary>Scenario 4: VCS-Driven Workflow with Pull Request Speculative Plans</summary>

**Situation**: An infrastructure team wants to enforce infrastructure code review. Before any Terraform change is applied to production, it must be reviewed via a GitHub pull request. The team wants Terraform plan output to be visible directly in the PR so reviewers don't need to clone and run locally.

**Setup**:

1. In HCP Terraform, connect the `production` workspace to the GitHub repository
2. Set the working directory to `environments/production`
3. Set the branch to `main`
4. Enable speculative plans for pull requests

**Developer Workflow**:

```bash
# Developer creates a feature branch
git checkout -b add-new-s3-bucket

# Makes changes to Terraform config
# ...

# Pushes and opens PR
git push origin add-new-s3-bucket
# Opens PR on GitHub
```

**What Happens Automatically**:

```
PR Opened
  ↓
HCP Terraform runs speculative plan (read-only, no apply possible)
  ↓
Plan result posted as GitHub status check on the PR:
  ✓ "Terraform Cloud — 1 to add, 0 to change, 0 to destroy"
  ↓
Reviewer sees exactly what will change BEFORE merging
  ↓
PR approved and merged to main
  ↓
HCP Terraform queues full plan + apply run
  ↓
Team lead approves apply in HCP UI
  ↓
Infrastructure change applied
```

**Expected Outcome**: Every infrastructure change goes through code review with plan output visible in GitHub. No developer can bypass the process by running locally — the workspace's execution mode is remote.

**Exam Sub-Objective**: Understand VCS-driven workflow and speculative plans (Objective 8).

</details>

---

<details><summary>Scenario 5: Orchestrating Multi-Workspace Deployments with Run Triggers</summary>

**Situation**: A company's production infrastructure is split into 3 workspaces that must be deployed in order: `networking` (VPCs, subnets) → `platform` (EKS cluster, IAM roles) → `application` (Kubernetes deployments, load balancers). When networking changes, the downstream workspaces should automatically update.

**Run Trigger Configuration** (HCP Terraform UI):

```
Workspace: platform-production
  Settings → Run Triggers → Source Workspaces:
    + networking-production   ← trigger platform when networking applies

Workspace: application-production  
  Settings → Run Triggers → Source Workspaces:
    + platform-production     ← trigger application when platform applies
```

**Resulting Deployment Chain**:

```
Developer merges networking change to main
  ↓
[networking-production] Plan → Apply (completes)
  ↓  (run trigger fires)
[platform-production] Run queued automatically → Plan → Apply (completes)
  ↓  (run trigger fires)
[application-production] Run queued automatically → Plan → Apply (completes)
```

**Combined with terraform_remote_state**:

```hcl
# platform workspace reads new subnet IDs from networking
data "terraform_remote_state" "networking" {
  backend = "remote"
  config = {
    organization = "acme-corp"
    workspaces   = { name = "networking-production" }
  }
}

resource "aws_eks_cluster" "main" {
  name = "production"
  vpc_config {
    subnet_ids = data.terraform_remote_state.networking.outputs.private_subnet_ids
  }
}
```

**Expected Outcome**: A single merge to the networking repo kicks off an automated, sequential deployment cascade across all three layers. Each workspace reads the latest outputs from the workspace above it via `terraform_remote_state`.

**Exam Sub-Objective**: Use run triggers to coordinate multi-workspace deployments; combine with `terraform_remote_state` for data flow (Objective 8).

</details>

---

## Quick Reference Summary

### `cloud` Block vs. `backend "remote"`

| | `cloud` block | `backend "remote"` |
|---|---|---|
| Since | Terraform 1.1 | Older |
| Status | **Preferred** | Deprecated for HCP |
| Tag-based workspaces | Yes | No |

### HCP Terraform Workspace Execution Modes

| Mode | Runs On | State In |
|---|---|---|
| Remote | HCP workers | HCP |
| Local | Your CLI | HCP |
| Agent | Your agent | HCP |

### Run Lifecycle

```
Trigger → Plan → Policy Check → Confirm → Apply → State Updated
```

### Key HCP Terraform Concepts

| Concept | Purpose |
|---|---|
| Variable sets | Reusable variable collections across workspaces |
| `terraform_remote_state` | Read outputs from another workspace |
| Run triggers | Auto-queue downstream workspace on upstream apply |
| Projects | Organize workspaces; project-level access control |
| Speculative plan | Read-only plan for PR review (no apply possible) |

### Authentication

```bash
terraform login                    # interactive browser flow
# Token stored: ~/.terraform.d/credentials.tfrc.json
# CI/CD: export TF_TOKEN_app_terraform_io="<token>"
```